# Compare models: Random Forest, XGBoost, and RNN

This notebook runs a simple comparison between three model types on well-log data:
- RandomForestRegressor (sklearn)
- XGBoost (XGBRegressor)
- RNN (Keras LSTM)

It assumes you have a `sample_wells.csv` in the working directory with columns: `well_id, depth, GR, RHOB, NPHI, DT, RES, azimuth, inclination, target`.
Adjust `FEATURE_COLS` and parameters as needed.


In [ ]:
# Requirements: pandas, numpy, scikit-learn, xgboost, tensorflow, matplotlib, joblib
import warnings
warnings.filterwarnings('ignore')
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import tensorflow as tf
import matplotlib.pyplot as plt
%matplotlib inline


In [ ]:
# Config
CSV_PATH = 'sample_wells.csv'  # change if needed
FEATURE_COLS = ['GR','RHOB','NPHI','DT','RES']
TARGET_COL = 'target'
WINDOW = 11  # for sequence model
RNN_EPOCHS = 5  # keep small for demo
RANDOM_STATE = 42


In [ ]:
# Utility: rolling-window features used by RF/XGB baseline
def make_features(df, feature_cols, window=5):
    X = df[feature_cols].copy()
    for col in feature_cols:
        X[f'{col}_mean_{window}'] = df[col].rolling(window, center=True, min_periods=1).mean()
        X[f'{col}_std_{window}'] = df[col].rolling(window, center=True, min_periods=1).std().fillna(0)
        X[f'{col}_grad'] = df[col].diff().fillna(0)
    return X.fillna(method='ffill').fillna(method='bfill').fillna(0)


In [ ]:
# Utility: sequence generator for RNN (per well)
def make_sequences(df, feature_cols, window=11, stride=1, label_pos='center'):
    arr = df[feature_cols].values
    n = len(arr)
    half = window // 2
    Xs, ys, idxs = [], [], []
    for i in range(0, n - window + 1, stride):
        seq = arr[i:i+window]
        if label_pos == 'center':
            label_idx = i + half
        else:
            label_idx = i + window - 1
        Xs.append(seq)
        ys.append(df.iloc[label_idx][TARGET_COL])
        idxs.append(df.index[label_idx])
    return np.array(Xs), np.array(ys), np.array(idxs)


In [ ]:
# Load data
df = pd.read_csv(CSV_PATH)
print('Rows:', len(df))
if 'well_id' not in df.columns:
    df['well_id'] = df.get('well', 'well_0')
# Quick sanity: drop rows without target
df = df.dropna(subset=[TARGET_COL])
print('After dropna target:', len(df))
df.head()


In [ ]:
# Split wells into train/test to avoid leakage
wells = df['well_id'].unique()
train_wells, test_wells = train_test_split(wells, test_size=0.2, random_state=RANDOM_STATE)
train_df = df[df['well_id'].isin(train_wells)].reset_index(drop=True)
test_df = df[df['well_id'].isin(test_wells)].reset_index(drop=True)
print('Train rows:', len(train_df), 'Test rows:', len(test_df))


## Baseline: Random Forest

In [ ]:
# Prepare features for RF/XGB
X_train = make_features(train_df, FEATURE_COLS, window=5)
X_test = make_features(test_df, FEATURE_COLS, window=5)
# align columns and fill
cols = X_train.columns.tolist()
X_test = X_test.reindex(columns=cols).fillna(0)
y_train = train_df[TARGET_COL].values
y_test = test_df[TARGET_COL].values
# scale
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
# RF
rf = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train_s, y_train)
rf_preds = rf.predict(X_test_s)
print('RF RMSE:', mean_squared_error(y_test, rf_preds, squared=False))
print('RF MAE:', mean_absolute_error(y_test, rf_preds))
print('RF R2:', r2_score(y_test, rf_preds))


## XGBoost

In [ ]:
xgb_model = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=RANDOM_STATE, n_jobs=-1)
xgb_model.fit(X_train_s, y_train, eval_set=[(X_test_s, y_test)], early_stopping_rounds=20, verbose=False)
xgb_preds = xgb_model.predict(X_test_s)
print('XGB RMSE:', mean_squared_error(y_test, xgb_preds, squared=False))
print('XGB MAE:', mean_absolute_error(y_test, xgb_preds))
print('XGB R2:', r2_score(y_test, xgb_preds))


## RNN (LSTM)

In [ ]:
# Build sequences for train and test (concatenate wells)
Xs_tr, ys_tr = [], []
for well, g in train_df.groupby('well_id'):
    g = g.sort_values('depth').reset_index(drop=True)
    Xs, ys, _ = make_sequences(g, FEATURE_COLS, window=WINDOW)
    if len(Xs)>0:
        Xs_tr.append(Xs); ys_tr.append(ys)
if len(Xs_tr)==0:
    raise RuntimeError('No training sequences generated for RNN')
Xtr = np.concatenate(Xs_tr, axis=0)
ytr = np.concatenate(ys_tr, axis=0)
# test sequences
Xs_te, ys_te = [], []
for well, g in test_df.groupby('well_id'):
    g = g.sort_values('depth').reset_index(drop=True)
    Xs, ys, _ = make_sequences(g, FEATURE_COLS, window=WINDOW)
    if len(Xs)>0:
        Xs_te.append(Xs); ys_te.append(ys)
Xte = np.concatenate(Xs_te, axis=0) if len(Xs_te)>0 else np.zeros((0,WINDOW,len(FEATURE_COLS)))
yte = np.concatenate(ys_te, axis=0) if len(ys_te)>0 else np.zeros((0,))
print('RNN samples train:', Xtr.shape[0], 'test:', Xte.shape[0])
# scale per feature channel
nsamples, w, nfeat = Xtr.shape
scaler_r = StandardScaler()
Xtr_flat = Xtr.reshape(-1, nfeat)
Xtr_s = scaler_r.fit_transform(Xtr_flat).reshape(nsamples, w, nfeat)
Xte_s = scaler_r.transform(Xte.reshape(-1, nfeat)).reshape(Xte.shape) if Xte.shape[0]>0 else Xte
# build simple LSTM model
model = tf.keras.Sequential([tf.keras.layers.Masking(mask_value=0.0, input_shape=(WINDOW, len(FEATURE_COLS))),
                                     tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
                                     tf.keras.layers.Dropout(0.2),
                                     tf.keras.layers.Dense(64, activation='relu'),
                                     tf.keras.layers.Dense(1)])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.fit(Xtr_s, ytr, validation_split=0.1, epochs=RNN_EPOCHS, batch_size=128, verbose=1)
rnn_preds = model.predict(Xte_s).ravel() if Xte_s.shape[0]>0 else np.array([])
if rnn_preds.size>0:
    print('RNN RMSE:', mean_squared_error(yte, rnn_preds, squared=False))
    print('RNN MAE:', mean_absolute_error(yte, rnn_preds))
    print('RNN R2:', r2_score(yte, rnn_preds))
else:
    print('No RNN test samples to evaluate')


## Summary Plot

In [ ]:
# Scatter predictions vs true for available models (use XGB/RF/rnn if available)
plt.figure(figsize=(6,6))
if len(rf_preds)>0:
    plt.scatter(y_test, rf_preds, s=6, alpha=0.6, label='RF')
if len(xgb_preds)>0:
    plt.scatter(y_test, xgb_preds, s=6, alpha=0.6, label='XGB')
if rnn_preds.size>0:
    # note: rnn preds correspond to sequence-based center points; alignment with y_test may differ
    plt.scatter(yte, rnn_preds, s=6, alpha=0.6, label='RNN')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--')
plt.xlabel('True')
plt.ylabel('Pred')
plt.legend()
plt.title('Pred vs True')
plt.show()
